In [ ]:
"""
LSST_SRD_Redshift_Distributions_and_Binning
Source sample only
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from lsst_galaxy_sample import LSSTGalaxySample as lsst


# ------------------------------------------------------------------
# Helper: replacement for cmr.take_cmap_colors
# ------------------------------------------------------------------
def take_cmap_colors(cmap_name, n, cmap_range=(0.0, 1.0), return_hex=True):
    cmap = plt.get_cmap(cmap_name)
    values = np.linspace(cmap_range[0], cmap_range[1], n)
    colors = cmap(values)
    if return_hex:
        return [mcolors.to_hex(c) for c in colors]
    return colors


# ------------------------------------------------------------------
# Define the redshift interval and forecast years
# ------------------------------------------------------------------
redshift_range = np.linspace(0.0, 3.5, 500)
forecast_years = ["1", "4", "7", "10"]
samples = {}

# ------------------------------------------------------------------
# Generate LSST SOURCE samples only
# ------------------------------------------------------------------
for year in forecast_years:
    init = lsst(year, redshift_range)
    samples[year] = {
        "source_nz": init.source_sample(normalized=True, save_file=True),
        "source_bins": init.source_bins(normalized=True, save_file=True),
        "source_bin_centers": init.source_bin_centers(decimal_places=3),
    }


# ------------------------------------------------------------------
# Plot styling
# ------------------------------------------------------------------
plt.rcParams.update({
    "lines.linewidth": 3,
    "axes.labelsize": 25,
    "axes.titlesize": 28,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 15
})


# ------------------------------------------------------------------
# Define colors for SOURCE plots only
# ------------------------------------------------------------------
colors = {
    "1": take_cmap_colors("inferno", len(samples["1"]["source_bins"]),
                          cmap_range=(0.2, 0.85)),
    "4": take_cmap_colors("inferno", len(samples["4"]["source_bins"]),
                          cmap_range=(0.2, 0.85)),
    "7": take_cmap_colors("inferno", len(samples["7"]["source_bins"]),
                          cmap_range=(0.2, 0.85)),
    "10": take_cmap_colors("inferno", len(samples["10"]["source_bins"]),
                           cmap_range=(0.2, 0.85)),
}


# ------------------------------------------------------------------
# Plot SOURCE redshift distributions
# ------------------------------------------------------------------
fig, ax = plt.subplots(1, 1, figsize=(6, 4.5))
lw = 4

ax.plot(redshift_range, samples["1"]["source_nz"],
        label="Y1", color=colors["1"][-2], lw=lw)
ax.plot(redshift_range, samples["4"]["source_nz"],
        label="Y4", color=colors["4"][-2], lw=lw)
ax.plot(redshift_range, samples["7"]["source_nz"],
        label="Y7", color=colors["7"][-2], lw=lw)
ax.plot(redshift_range, samples["10"]["source_nz"],
        label="Y10", color=colors["10"][2], lw=lw)

ax.set_title("LSST Source Sample\nnormalized")
ax.set_xlabel("redshift")
ax.set_ylabel("redshift distribution")
ax.tick_params(direction="in")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Plot SOURCE tomographic bins
# ------------------------------------------------------------------
years = ["1", "4", "7", "10"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
lw = 4

for idx, year in enumerate(years):
    ax = axes[idx]
    for bin_idx, (key, data) in enumerate(samples[year]["source_bins"].items()):
        ax.plot(
            redshift_range,
            data,
            label=f"bin {key + 1}",
            color=colors[year][bin_idx],
            lw=lw
        )

    ax.set_xlabel("redshift")
    ax.set_ylabel("redshift distribution")
    ax.set_title(f"Source bins for LSST Y{year}")
    ax.tick_params(direction="in")
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------------
# Print SOURCE bin centers
# ------------------------------------------------------------------
print(f"Source bin centers for Y1: {samples['1']['source_bin_centers']}")
print(f"Source bin centers for Y4: {samples['4']['source_bin_centers']}")
print(f"Source bin centers for Y7: {samples['7']['source_bin_centers']}")
print(f"Source bin centers for Y10: {samples['10']['source_bin_centers']}")

print("----------------------------------------")

for year in years:
    centers = list(samples[year]["source_bin_centers"].values())
    print(f"Source bin centers for Y{year}: {centers}")


# ------------------------------------------------------------------
# Plot SOURCE bins with bin centers marked
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()
lw = 4

for idx, year in enumerate(years):
    ax = axes[idx]

    global_max = max([max(data) for data in samples[year]["source_bins"].values()])
    ax.set_ylim(0, global_max * 1.1)

    for bin_idx, (key, data) in enumerate(samples[year]["source_bins"].items()):
        center = samples[year]["source_bin_centers"][key]
        max_bin_value = max(data)

        ax.axvline(
            center,
            color=colors[year][bin_idx],
            linestyle="--",
            lw=2,
            ymin=0,
            ymax=max_bin_value / (global_max * 1.1)
        )

        ax.plot(
            redshift_range,
            data,
            label=rf"$z_\mathrm{{center}}^{key+1}={center:.3f}$",
            color=colors[year][bin_idx],
            lw=lw
        )

    ax.set_xlabel("redshift")
    ax.set_ylabel("redshift distribution")
    ax.set_title(f"Source bins for LSST Y{year}")
    ax.tick_params(direction="in")
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()